In [1]:
%pip install -q -U transformers accelerate safetensors tqdm

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/materials-qa-evaluation")
INPUT_PATH = PROJECT_DIR / "data/final/qa_dataset.jsonl"
OUTPUT_PATH = PROJECT_DIR / "data/evaluation/qwen_results.jsonl"

INPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_PATH)

Input: /content/drive/MyDrive/materials-qa-evaluation/data/final/qa_dataset.jsonl
Output: /content/drive/MyDrive/materials-qa-evaluation/data/evaluation/qwen_results.jsonl


In [6]:
from google.colab import files

uploaded = files.upload()

if "qa_dataset.jsonl" not in uploaded:
    raise ValueError("Please upload qa_dataset.jsonl")

INPUT_PATH.write_bytes(uploaded["qa_dataset.jsonl"])

print("Dataset saved to:", INPUT_PATH)

Saving qa_dataset.jsonl to qa_dataset.jsonl
Dataset saved to: /content/drive/MyDrive/materials-qa-evaluation/data/final/qa_dataset.jsonl


In [7]:
import json

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

records = read_jsonl(INPUT_PATH)

print("Records:", len(records))
print("First QA ID:", records[0]["qa_id"])
print("First question:", records[0]["question"])
print("Has abstract:", bool(records[0]["source_abstract"]))

assert len(records) == 234
assert all(record.get("source_abstract") for record in records)

Records: 234
First QA ID: qa_0001
First question: What is the material property that changes from parabolic to cubic-like behavior under high pressure in transition metals?
Has abstract: True


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model.eval()

print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cuda:0


In [9]:
SYSTEM_PROMPT = """
You answer materials-science questions using only the supplied abstract.

Do not use outside knowledge or make unsupported inferences.
If the abstract does not contain enough information to answer the question,
respond exactly: Not stated

Otherwise, provide a concise answer based only on the abstract.
""".strip()


def build_messages(record):
    user_prompt = f"""
Abstract:
{record["source_abstract"]}

Question:
{record["question"]}
""".strip()

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

In [10]:
def generate_answer(record):
    messages = build_messages(record)

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    prompt_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][prompt_length:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return answer

In [11]:
test_record = records[0]

print("Question:")
print(test_record["question"])

print("\nExpected answer:")
print(test_record["answer"])

print("\nQwen answer:")
print(generate_answer(test_record))

Question:
What is the material property that changes from parabolic to cubic-like behavior under high pressure in transition metals?

Expected answer:
atomic volume

Qwen answer:
Atomic volume


In [12]:
import os
from tqdm.auto import tqdm


def save_jsonl_safely(path, rows):
    temporary_path = path.with_suffix(".tmp")

    with open(temporary_path, "w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")

    os.replace(temporary_path, path)


if OUTPUT_PATH.exists():
    results = read_jsonl(OUTPUT_PATH)
else:
    results = []

completed_ids = {result["qa_id"] for result in results}

print("Already completed:", len(completed_ids))
print("Remaining:", len(records) - len(completed_ids))


for record in tqdm(records):
    if record["qa_id"] in completed_ids:
        continue

    model_answer = generate_answer(record)

    result = {
        "qa_id": record["qa_id"],
        "arxiv_id": record["arxiv_id"],
        "question_type": record["question_type"],
        "source_abstract": record["source_abstract"],
        "question": record["question"],
        "expected_answer": record["answer"],
        "supporting_passage": record["supporting_passage"],
        "is_answerable": record["is_answerable"],
        "model_id": MODEL_ID,
        "prompt_version": "grounded_qa_v1",
        "model_answer": model_answer,
        "evaluation_label": None,
        "evaluation_notes": "",
    }

    results.append(result)
    completed_ids.add(record["qa_id"])

    save_jsonl_safely(OUTPUT_PATH, results)


print("Completed:", len(results))
print("Saved to:", OUTPUT_PATH)

Already completed: 0
Remaining: 234


  0%|          | 0/234 [00:00<?, ?it/s]

Completed: 234
Saved to: /content/drive/MyDrive/materials-qa-evaluation/data/evaluation/qwen_results.jsonl


In [13]:
results = read_jsonl(OUTPUT_PATH)

print("Results:", len(results))
print("Unique QA IDs:", len({row["qa_id"] for row in results}))
print("Empty answers:", sum(not row["model_answer"] for row in results))

assert len(results) == 234
assert len({row["qa_id"] for row in results}) == 234
assert all(row["model_answer"] for row in results)

print("\nExample result:")
print(json.dumps(results[0], indent=2, ensure_ascii=False))

Results: 234
Unique QA IDs: 234
Empty answers: 0

Example result:
{
  "qa_id": "qa_0001",
  "arxiv_id": "2609.08197v1",
  "question_type": "material_property",
  "source_abstract": "We investigated the evolution of the well-known parabolic dependence of atomic volume on atomic number in transition metals under extreme compression at pressures up to 400 GPa using density functional theory calculations. Our results reveal that the ambient-pressure parabolic trend transforms into a characteristic cubic-like behavior at high pressures. This evolution is attributed to the higher compressibility of bcc transition metals associated with comparatively large increases in the total energy. The present findings are discussed in relation to previous experimental observations and first-principles calculations.",
  "question": "What is the material property that changes from parabolic to cubic-like behavior under high pressure in transition metals?",
  "expected_answer": "atomic volume",
  "supporti

In [14]:
from google.colab import files

files.download(str(OUTPUT_PATH))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>